# Familia 3 · Ingeniería del prompt (*prompting*)

*lo que se le dice al modelo, sin tocar sus pesos*

**PGL · el proyecto · cuaderno de arranque.** El proyecto es **una capa** de un
modelo de lenguaje grande (*large language model*, LLM) moderno, sobre un modelo de verdad, en español: la misma tarea **sin la capa
y con ella**, medidas igual sobre **30 a 50 casos** con su respuesta correcta
escrita antes, y los fallos explicados. Este cuaderno hace todo eso con diez casos
de ejemplo. Vuestro trabajo empieza cambiándolos por los vuestros: la última
sección dice cómo, paso a paso.

| | |
|---|---|
| **Lo que entra** | una reseña de un restaurante (la misma tarea que la familia 2, para poder comparar) |
| **Lo que sale** | su clase: positiva, negativa o mixta |
| **Sin la capa** | la pregunta tal cual |
| **Con la capa** | un **mensaje de sistema** (*system prompt*) que define cada clase, **ejemplos resueltos** (*few-shot*) y **un formato de salida fijo** |
| **La medida** | la **exactitud**: el % de reseñas con la clase correcta |
| **En clase** | la plantilla de chat, en la sesión del post-entrenamiento; el prompting se verá en su propia sesión |

## 0. Preparación, en Colab

Este cuaderno se ejecuta en **Google Colab** (colab.research.google.com), en CPU:
no hace falta cambiar el tipo de entorno (con un modelo mayor, sí: **Entorno de
ejecución → Cambiar tipo de entorno → GPU T4**).

1. Descomprimid en vuestro ordenador el zip de esta familia, `familia-3-prompt.zip`
   (del Campus): trae este cuaderno y la carpeta `datos/`.
2. En Colab: **Archivo → Subir cuaderno** → `familia-3-prompt.ipynb`.
3. En Colab, el icono de la **carpeta**, a la izquierda → **Subir** → el zip
   entero, `familia-3-prompt.zip`. Esperad a que aparezca en la lista.
4. Ejecutad la celda de abajo: instala lo que Colab no trae, descomprime el zip y
   dice si está cada fichero que el cuaderno lee.

Si Colab se desconecta, borra los ficheros: volved a subir el zip y a ejecutar
esta celda. **La primera vez se descarga el modelo** (un giga). Hugging Face puede
avisar de que no hay token (`HF_TOKEN`): es sólo un aviso; cómo ponerlo, en las
láminas de Colab de la sesión de la atención (martes 27 de octubre, la de las
cabezas apagadas).

In [1]:
import glob
import os
import sys
import zipfile

# Descomprimir el zip de la familia, se llame como se llame
# (el navegador a veces le añade « (1)» al nombre).
if not os.path.exists('datos'):
    for zip_subido in glob.glob('*.zip'):
        zipfile.ZipFile(zip_subido).extractall('.')
        print('Descomprimido:', zip_subido)

print('Carpeta de trabajo:', os.getcwd())
FICHEROS_QUE_LEE = ['datos/casos.tsv', 'datos/ejemplos.tsv']
for fichero in FICHEROS_QUE_LEE:
    print('  está  ' if os.path.exists(fichero) else '  FALTA ', fichero)

Carpeta de trabajo: /home/user/NLPCourses/master-nlp/proyecto/arranque/familia-3-prompt
  está   datos/casos.tsv
  está   datos/ejemplos.tsv


## 1. Los casos: `datos/casos.tsv`, y `datos/ejemplos.tsv`

Reseñas de restaurantes, inventadas, cada una con su clase puesta a mano:
**positiva**, **negativa** o **mixta** (algo bueno y algo malo). Columnas `resena`
y `clase`. Diez para medir (`CASOS`), y treinta aparte en `datos/ejemplos.tsv`
(`EJEMPLOS`) de donde salen los ejemplos que se ponen en el prompt: **nunca se mide con un caso que esté en el prompt**.

`leer_tsv(ruta)` lee un fichero **separado por tabuladores** (`.tsv`, se abre con
cualquier hoja de cálculo): la primera línea dice las columnas, y cada línea
siguiente es un caso. Devuelve una lista con un diccionario por línea,
`{columna: valor}`; los valores llegan como texto.

In [2]:
import csv


def leer_tsv(ruta: str) -> list:
    """Las filas de un fichero separado por tabuladores, cada una como {columna: valor}."""
    with open(ruta, encoding='utf-8', newline='') as fichero:
        return list(csv.DictReader(fichero, delimiter='\t'))


CASOS = [(fila['resena'], fila['clase']) for fila in leer_tsv('datos/casos.tsv')]
EJEMPLOS = [(fila['resena'], fila['clase']) for fila in leer_tsv('datos/ejemplos.tsv')]
CLASES = ['positiva', 'negativa', 'mixta']
print(f'{len(CASOS)} casos para medir, {len(EJEMPLOS)} aparte')
print('  ', CASOS[0])

10 casos para medir, 30 aparte
   ('La paella, fría y sin sabor.', 'negativa')


## 2. La medida, antes de nada

La **exactitud**: el % de reseñas con la clase correcta. El modelo contesta con
frases, no con una etiqueta, así que primero hay que sacar la clase de su
respuesta:

- `clase_de(respuesta)` busca en la respuesta las tres clases y devuelve **la que
  aparece primero**, o nada si no aparece ninguna;
- `misma_clase(respuesta, correcta)` dice si esa clase es la correcta.

Es una decisión de medida, y se puede discutir: una respuesta que dice «no es
negativa, es positiva» se lee como negativa. En vuestro proyecto, escribid la
vuestra antes de mirar las respuestas.

In [3]:
def clase_de(respuesta: str) -> str:
    """La primera de las tres clases que aparece en la respuesta, o '' si ninguna."""
    texto = respuesta.lower()
    posiciones = {clase: texto.find(clase) for clase in CLASES if clase in texto}
    if not posiciones:
        return ''
    return min(posiciones, key=posiciones.get)


def misma_clase(respuesta: str, correcta: str) -> bool:
    """Verdadero si la clase que dice la respuesta es la correcta."""
    return clase_de(respuesta) == correcta


print(clase_de('Diría que es mixta: la comida, bien; el servicio, mal.'))

mixta


## 3. Sin la capa: la pregunta, tal cual

**El modelo**: `Qwen2.5-0.5B-Instruct`, un modelo de instrucciones de unos 500
millones de pesos (el de la sesión del post-entrenamiento), que corre en el Colab
gratuito sin GPU. Es pequeño a propósito: sus fallos se ven, y la capa tiene dónde
mejorar. Una mesa que quiera uno mayor cambia `NOMBRE_DEL_MODELO` (por ejemplo,
`Qwen/Qwen2.5-1.5B-Instruct`, con la GPU de Colab).

La celda de abajo:
1. descarga el modelo y su tokenizador;
2. lo carga en `float32`: se guardó en `bfloat16`, un formato de números que las
   CPU de Colab hacen muy despacio (lo que pasó en la sesión de la atención);
3. define `preguntar(mensajes, tokens_nuevos)`: le da al modelo una conversación
   (una lista de mensajes, cada uno `{'role': ..., 'content': ...}`), con su
   plantilla de chat, y devuelve su respuesta. Elige siempre el token más probable
   (**descodificación voraz**), así que la misma pregunta da siempre la misma
   respuesta; `tokens_nuevos` es lo más larga que puede ser;
4. define `medir(responder, casos, es_correcta)`: pasa cada caso por `responder`
   (una función que recibe la entrada del caso y devuelve una respuesta), comprueba
   cada respuesta con `es_correcta`, y devuelve la exactitud y, caso a caso, la
   entrada, la respuesta, la correcta y si acierta. **Es la misma con la capa y
   sin ella**: por eso se pueden comparar;
5. define `ensenar(detalle, cuantos)`: imprime los primeros casos de ese detalle.

Después, `sin_la_capa(resena)` le hace la pregunta `PREGUNTA_SIN` y nada más.

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging

logging.set_verbosity_error()          # que sólo hable si algo falla

NOMBRE_DEL_MODELO = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizador = AutoTokenizer.from_pretrained(NOMBRE_DEL_MODELO)
modelo = AutoModelForCausalLM.from_pretrained(NOMBRE_DEL_MODELO, dtype=torch.float32)
modelo.eval()


def preguntar(mensajes: list, tokens_nuevos: int) -> str:
    """La respuesta del modelo a una conversación, eligiendo siempre el token más probable."""
    entrada = tokenizador.apply_chat_template(mensajes, add_generation_prompt=True,
                                              return_tensors='pt', return_dict=True)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=tokens_nuevos, do_sample=False)
    largo_de_la_pregunta = entrada['input_ids'].shape[1]
    return tokenizador.decode(salida[0][largo_de_la_pregunta:], skip_special_tokens=True).strip()


def medir(responder, casos: list, es_correcta) -> tuple:
    """(exactitud en %, [(entrada, respuesta, correcta, ¿acierta?), ...]) sobre los casos."""
    detalle = []
    for entrada, correcta in casos:
        respuesta = responder(entrada)
        detalle.append((entrada, respuesta, correcta, es_correcta(respuesta, correcta)))
    aciertos = sum(1 for *_, acierta in detalle if acierta)
    return 100 * aciertos / len(casos), detalle


def ensenar(detalle: list, cuantos: int) -> None:
    """Los primeros `cuantos` casos: lo que entró, lo que respondió, lo correcto, y si acierta."""
    for entrada, respuesta, correcta, acierta in detalle[:cuantos]:
        marca = 'bien' if acierta else 'MAL '
        print(f'{marca}  {str(entrada)[:70]!r}')
        print(f'      respondió: {str(respuesta)[:90]!r}')
        print(f'      correcta:  {correcta!r}')

In [5]:
PREGUNTA_SIN = 'Clasifica esta reseña: '
TOKENS_NUEVOS = 60          # la pregunta sola da respuestas largas: que quepa la clase


def sin_la_capa(resena: str) -> str:
    """Sólo la pregunta, sin más."""
    return preguntar([{'role': 'user', 'content': PREGUNTA_SIN + resena}], TOKENS_NUEVOS)


exactitud_sin, detalle_sin = medir(sin_la_capa, CASOS, misma_clase)
print(f'sin la capa: {exactitud_sin:.0f} %')
print()
ensenar(detalle_sin, 4)

sin la capa: 70 %

bien  'La paella, fría y sin sabor.'
      respondió: 'La reseña "La paella, fría y sin sabor" clasificada como negativa sería:\n\nNegativa\n\nEsta r'
      correcta:  'negativa'
bien  'Un lugar encantador, con un personal muy atento.'
      respondió: 'Esta reseña clasificable como positiva. La descripción de "Un lugar encantador" sugiere qu'
      correcta:  'positiva'
MAL   'La carne era excelente, pero esperamos cuarenta minutos.'
      respondió: 'La clasificación de esta reseña sería:\n\n1. Positiva (positivo)\n2. Basada en el deseo del c'
      correcta:  'mixta'
bien  'No volveré: el pescado no era fresco.'
      respondió: 'Esta reseña clasificada como negativa o desaprobación sería:\n\n"El pescado no era fresco."\n'
      correcta:  'negativa'


## 4. Con la capa: sistema, ejemplos y formato

- `SISTEMA`: el **mensaje de sistema**, que va antes de la conversación: qué hace
  el modelo, qué es cada clase, y que conteste con una sola palabra.
- `ejemplos_del_prompt(ejemplos, cuantos_por_clase)` toma los primeros
  `cuantos_por_clase` de cada clase de `EJEMPLOS` y los convierte en turnos de la
  conversación: el usuario dice la reseña, el asistente contesta con su clase. Así
  el modelo ve el formato que se espera.
- `con_la_capa(resena)`: el sistema, los turnos de ejemplo, y la reseña.

In [6]:
SISTEMA = ('Clasificas reseñas de restaurantes. Respondes con una sola palabra: '
           'positiva (todo lo que dice es bueno), negativa (todo es malo) '
           'o mixta (dice algo bueno y algo malo).')
CUANTOS_EJEMPLOS_POR_CLASE = 1


def ejemplos_del_prompt(ejemplos: list, cuantos_por_clase: int) -> list:
    """Los primeros `cuantos_por_clase` de cada clase, como turnos de la conversación."""
    turnos = []
    for clase in CLASES:
        de_esta_clase = [resena for resena, su_clase in ejemplos if su_clase == clase]
        for resena in de_esta_clase[:cuantos_por_clase]:
            turnos.append({'role': 'user', 'content': resena})
            turnos.append({'role': 'assistant', 'content': clase})
    return turnos


TURNOS_DE_EJEMPLO = ejemplos_del_prompt(EJEMPLOS, CUANTOS_EJEMPLOS_POR_CLASE)


def con_la_capa(resena: str) -> str:
    """Sistema, los ejemplos resueltos y la reseña."""
    mensajes = ([{'role': 'system', 'content': SISTEMA}] + TURNOS_DE_EJEMPLO
                + [{'role': 'user', 'content': resena}])
    return preguntar(mensajes, TOKENS_NUEVOS)


exactitud_con, detalle_con = medir(con_la_capa, CASOS, misma_clase)
print(f'{"":14}{"exactitud":>10}')
print(f'{"sin la capa":14}{exactitud_sin:>9.0f}%')
print(f'{"con la capa":14}{exactitud_con:>9.0f}%')

               exactitud
sin la capa          70%
con la capa          70%


## 5. Las dos, medidas igual: los fallos, y si la diferencia es real

`cambios(detalle_sin, detalle_con)` pone los dos detalles lado a lado y separa los
casos en cuatro grupos: los que **arregla** la capa (mal sin ella, bien con ella),
los que **estropea** (al revés), los que fallan **en las dos** y los que aciertan
en las dos. Los dos primeros grupos son la diferencia; los fallos de los dos
últimos son los que tenéis que explicar.

**¿Es real la diferencia?** Con diez casos, cada caso vale diez puntos. El
**remuestreo pareado** (*bootstrap*; se explica aquí, y vuelve en la sesión de la clínica): sacar al
azar, **con repetición**, tantos casos como tenéis; contar los aciertos de cada
versión **en los mismos casos**; repetirlo mil veces. `remuestreo(aciertos_sin,
aciertos_con, veces, semilla)` devuelve en qué porcentaje de esas mil muestras gana
la versión con la capa, y en cuál empatan. Por encima de 95, la diferencia es creíble; por debajo,
se dice así en la defensa.

Lo difícil de esta familia es **saber si un cambio del prompt mejora de verdad**, o
sólo en los casos que se miraron: el remuestreo es la herramienta.

In [7]:
import random


def cambios(detalle_sin: list, detalle_con: list) -> dict:
    """Los casos, en cuatro grupos según aciertan sin la capa y con ella."""
    grupos = {'arregla': [], 'estropea': [], 'fallan las dos': [], 'aciertan las dos': []}
    for (entrada, resp_sin, correcta, bien_sin), (_, resp_con, _, bien_con) in zip(detalle_sin, detalle_con):
        if bien_con and not bien_sin:
            grupo = 'arregla'
        elif bien_sin and not bien_con:
            grupo = 'estropea'
        elif not bien_sin:
            grupo = 'fallan las dos'
        else:
            grupo = 'aciertan las dos'
        grupos[grupo].append((entrada, resp_sin, resp_con, correcta))
    return grupos


def remuestreo(aciertos_sin: list, aciertos_con: list, veces: int, semilla: int) -> tuple:
    """(% de muestras en que gana «con», % en que empatan); cada muestra, los mismos casos para las dos."""
    azar = random.Random(semilla)
    cuantos = len(aciertos_sin)
    gana = empata = 0
    for _ in range(veces):
        muestra = [azar.randrange(cuantos) for _ in range(cuantos)]   # con repetición
        con = sum(aciertos_con[i] for i in muestra)
        sin = sum(aciertos_sin[i] for i in muestra)
        gana += con > sin
        empata += con == sin
    return 100 * gana / veces, 100 * empata / veces

In [8]:
grupos = cambios(detalle_sin, detalle_con)
for grupo, casos_del_grupo in grupos.items():
    print(f'{grupo}: {len(casos_del_grupo)}')
print()
for entrada, resp_sin, resp_con, correcta in grupos['estropea'] + grupos['fallan las dos']:
    print(f'{str(entrada)[:70]!r}')
    print(f'    sin: {str(resp_sin)[:80]!r}')
    print(f'    con: {str(resp_con)[:80]!r}')
    print(f'    correcta: {correcta!r}')

VECES = 1000
SEMILLA = 0
aciertos_sin = [int(acierta) for *_, acierta in detalle_sin]
aciertos_con = [int(acierta) for *_, acierta in detalle_con]
gana, empata = remuestreo(aciertos_sin, aciertos_con, VECES, SEMILLA)
print()
print(f'{VECES} muestras de {len(aciertos_sin)} casos: la capa gana en el {gana:.0f} %, '
      f'empata en el {empata:.0f} %, pierde en el {abs(100 - gana - empata):.0f} %')

arregla: 0
estropea: 0
fallan las dos: 3
aciertan las dos: 7

'La carne era excelente, pero esperamos cuarenta minutos.'
    sin: 'La clasificación de esta reseña sería:\n\n1. Positiva (positivo)\n2. Basada en el d'
    con: 'negativa'
    correcta: 'mixta'
'Precios altos para lo que ofrecen, aunque el vino era bueno.'
    sin: 'Esta reseña clasificada como "Baja" o "Mediana". La reseña sugiere que la calida'
    con: 'negativa'
    correcta: 'mixta'
'El ambiente, estupendo; la comida, regular.'
    sin: 'La clasificación de esta reseña sería:\n\n1. Positiva (positivo)\n2. Basada en una '
    con: 'positiva'
    correcta: 'mixta'

1000 muestras de 10 casos: la capa gana en el 0 %, empata en el 100 %, pierde en el 0 %


## 6. El segundo anotador: cuánto se fía uno de las respuestas correctas

Las respuestas correctas las escribe una persona, y dos personas no siempre
coinciden. **El acuerdo entre anotadores** es el porcentaje de casos en que dos
personas, cada una por su cuenta y sin mirar lo de la otra, dan la misma
respuesta. Es **el techo**: ninguna versión puede medirse por encima de lo que las
personas coinciden entre sí.

**Cómo se hace en el proyecto**: otro miembro de la mesa escribe la respuesta
correcta de **veinte casos** por su cuenta, con las mismas instrucciones escritas;
se calcula el acuerdo con la función de abajo; donde no coincidís, se decide y se
aclaran las instrucciones. El número y los desacuerdos van a la defensa.

In [9]:
def acuerdo(respuestas_1: list, respuestas_2: list) -> float:
    """% de casos en que dos anotadores dan la misma respuesta (las dos listas, en el mismo orden)."""
    iguales = sum(1 for una, otra in zip(respuestas_1, respuestas_2) if una == otra)
    return 100 * iguales / len(respuestas_1)


# Un ejemplo: dos personas, cinco casos; coinciden en cuatro.
print(acuerdo(['a', 'b', 'a', 'c', 'a'], ['a', 'b', 'c', 'c', 'a']), '%')

80.0 %


## 7. Los mandos: qué cambiar, y qué esperáis que pase

Cada parámetro que cambia el resultado está declarado en una celda, con su valor.
Para cada uno: **escribid antes qué esperáis**, cambiadlo y volved a ejecutar desde
su celda hasta el final.

| Dónde | Cambiad | Antes de ejecutar, pensad |
|---|---|---|
| `CUANTOS_EJEMPLOS_POR_CLASE` | de 1 a 0 (sólo sistema), y a 3 | ¿ayudan más los ejemplos o la definición? |
| `SISTEMA` | quitad las definiciones entre paréntesis | ¿qué reseñas mixtas se pierden? |
| `TOKENS_NUEVOS` | de 60 a 3 | ¿qué pasa sin la capa, que contestaba con frases? |

---

## 8. Vuestro proyecto, paso a paso

1. **Ejecutad este cuaderno tal cual** (**Entorno de ejecución → Ejecutar todo**)
   y mirad los fallos de §5.
2. **Cambiad los casos por los vuestros.** Abrid `datos/casos.tsv` con una hoja de
   cálculo, borrad los de ejemplo y escribid **entre 5 y 10 casos vuestros**,
   cada uno con su texto y su respuesta, y unos ejemplos aparte en `datos/ejemplos.tsv`, con las mismas columnas. Guardadlo como texto separado por
   tabuladores, con el mismo nombre, subidlo a Colab y ejecutad todo otra vez.
3. **Los casos completos: entre 30 y 50**, con la respuesta correcta **escrita
   antes** de probar la capa, y unas instrucciones de cómo se decide cada una.
   Que haya casos difíciles: con casos fáciles, «sin» y «con» empatan. Veinte,
   también por un segundo miembro de la mesa (§6).
4. **La capa, mejorada** sobre lo que hace este cuaderno: es el trabajo de la mesa.
   Repartidlo: la parte de cada uno, escrita en el cuaderno.
5. **Las dos versiones, medidas igual** sobre todos los casos; los fallos
   clasificados (no basta el número: por qué falla cada una, con ejemplos) y el
   remuestreo de §5.
6. **Para la clínica y la defensa**: el problema y la línea base, lo que habéis
   añadido, la medida y si la diferencia es real, tres fallos explicados.

**Ideas**: extraer datos de mensajes; clasificar con muchas clases; pedir que razone paso a paso, y medir si ayuda. Para buscar: few-shot prompting, chain-of-thought, prompt sensitivity.

Las dudas, en el foro del proyecto del Campus. El cuaderno es vuestro: cambiad lo
que haga falta.